# TopicGPT Tuning: c-TF-IDF & Topic Filtering Optimization

This notebook tunes the c-TF-IDF and topic filtering parameters for TopicGPT's
assignment stage. It **reuses** existing assignment checkpoints from the modeling
phase and grid-searches over:

- **`max_df`** — c-TF-IDF maximum document frequency threshold
- **`min_docs`** — Minimum documents per topic to keep

Each subject uses its best sentence transformer model identified during modeling.

In [1]:
import os
import gc
import pickle
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import product
from tqdm import tqdm
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from sklearn.feature_extraction.text import TfidfVectorizer
from itertools import combinations
import warnings
import rbo

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## Configuration

In [2]:
LIST_SUBJECT = ["cs", "math", "physics"]
VERSION = "v1"

BASE_DIR = Path("../../../../data/preprocess")
CHECKPOINT_DIR = Path("../../../../models/topicGpt")
RESULT_DIR = Path("../../../../results/topicGpt/tunning")
TUNING_CKPT_DIR = CHECKPOINT_DIR  # tuning checkpoints go alongside modeling ones

# Best embedding model per subject (from modeling phase)
BEST_MODEL_MAP = {
    "cs":      "sentence_transformers_all_MiniLM_L6_v2",
    "math":    "sentence_transformers_all_MiniLM_L6_v2",
    "physics":  "all_distilroberta_v1",
}

# --- Tuning grid ---
MAX_DF_VALUES   = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
MIN_DOCS_VALUES = [1, 10, 25, 50, 100, 150, 200, 250, 300]

TOP_N_WORDS = 10
RBO_P = 0.9

# Create output dirs
for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)

print(f"Subjects: {LIST_SUBJECT}")
print(f"max_df grid:   {MAX_DF_VALUES}")
print(f"min_docs grid: {MIN_DOCS_VALUES}")
print(f"Total combos per subject: {len(MAX_DF_VALUES) * len(MIN_DOCS_VALUES)}")
print(f"Results dir: {RESULT_DIR.resolve()}")

Subjects: ['cs', 'math', 'physics']
max_df grid:   [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
min_docs grid: [1, 10, 25, 50, 100, 150, 200, 250, 300]
Total combos per subject: 54
Results dir: /home/nedo/Kuliah/TA/Program/results/topicGpt/tunning


## Checkpoint Utilities

In [3]:
def save_checkpoint(data, name: str, subject: str):
    """Save checkpoint to disk."""
    path = TUNING_CKPT_DIR / subject / f"{name}.pkl"
    with open(path, "wb") as f:
        pickle.dump(data, f)
    print(f"  Checkpoint saved: {path}")

def load_checkpoint(name: str, subject: str):
    """Load checkpoint from disk, return None if not found."""
    path = TUNING_CKPT_DIR / subject / f"{name}.pkl"
    if path.exists():
        with open(path, "rb") as f:
            data = pickle.load(f)
        print(f"  Checkpoint loaded: {path}")
        return data
    return None

## Data Loading

In [4]:
def load_dataset(subject: str) -> pd.DataFrame:
    """Load preprocessed dataset."""
    file_path = BASE_DIR / subject / "emb" / f"{VERSION}.csv"
    df = pd.read_csv(file_path)
    return df

all_data = {}
for subject in LIST_SUBJECT:
    df = load_dataset(subject)
    all_data[subject] = df
    print(f"{subject}: {len(df):,} documents loaded")

print(f"\nAll subjects loaded.")

cs: 165,756 documents loaded
math: 157,085 documents loaded
physics: 146,311 documents loaded

All subjects loaded.


## Load Modeling Checkpoints

Load the best assignment checkpoint per subject from the modeling phase.
These contain the base `assignment_df` produced by the best sentence transformer.

In [5]:
all_base_assignments = {}

for subject in LIST_SUBJECT:
    best_model = BEST_MODEL_MAP[subject]
    ckpt = load_checkpoint(f"assignment_{best_model}", subject)
    if ckpt is None:
        print(f"  ERROR: No checkpoint found for {subject}/{best_model}")
        continue

    all_base_assignments[subject] = ckpt["assignment_df"]
    n_docs = len(ckpt["assignment_df"])
    n_topics = ckpt["assignment_df"]["topic_id"].nunique()
    print(f"  {subject}: {n_docs:,} assignments, {n_topics} unique topics")
    print(f"  Modeling metrics: C_v={ckpt['metrics']['coherence']:.4f}  "
          f"IRBO={ckpt['metrics']['irbo']:.4f}  TQ={ckpt['metrics']['topic_quality']:.4f}")

print(f"\nLoaded {len(all_base_assignments)}/{len(LIST_SUBJECT)} subjects.")

  Checkpoint loaded: ../../../../models/topicGpt/cs/assignment_sentence_transformers_all_MiniLM_L6_v2.pkl
  cs: 65,516 assignments, 138 unique topics
  Modeling metrics: C_v=0.7095  IRBO=0.9893  TQ=0.8264
  Checkpoint loaded: ../../../../models/topicGpt/math/assignment_sentence_transformers_all_MiniLM_L6_v2.pkl
  math: 46,626 assignments, 124 unique topics
  Modeling metrics: C_v=0.7162  IRBO=0.9835  TQ=0.8288
  Checkpoint loaded: ../../../../models/topicGpt/physics/assignment_all_distilroberta_v1.pkl
  physics: 28,507 assignments, 74 unique topics
  Modeling metrics: C_v=0.7406  IRBO=0.9835  TQ=0.8449

Loaded 3/3 subjects.


## Tuning Functions

In [6]:
def filter_and_reindex_topics(df_assign: pd.DataFrame, min_docs: int) -> pd.DataFrame:
    """Filter topics with fewer than min_docs documents and reindex."""
    df_filtered = df_assign[df_assign["topic_id"] != -1].copy()

    topic_counts = df_filtered["topic_id"].value_counts()
    valid_topics = topic_counts[topic_counts >= min_docs].index

    df_filtered = df_filtered[df_filtered["topic_id"].isin(valid_topics)].copy()

    unique_topics = sorted(df_filtered["topic_id"].unique())
    topic_mapping = {old_id: new_idx for new_idx, old_id in enumerate(unique_topics)}

    df_filtered["original_topic_id"] = df_filtered["topic_id"]
    df_filtered["topic_id"] = df_filtered["topic_id"].map(topic_mapping)

    return df_filtered


def compute_topic_words_ctfidf(assignment_df: pd.DataFrame, texts: list,
                                max_df: float = 0.8, top_n: int = 10) -> dict:
    """Extract topic words using c-TF-IDF with parametric max_df."""
    topic_docs = []
    tids = []

    for tid, grp in assignment_df.groupby("topic_id"):
        if tid == -1:
            continue
        combined_text = " ".join([texts[i] for i in grp["doc_idx"].tolist() if i < len(texts)])
        topic_docs.append(combined_text)
        tids.append(tid)

    if not topic_docs:
        return {}

    vec = TfidfVectorizer(
        stop_words="english",
        max_features=10000,
        ngram_range=(1, 2),
        max_df=max_df
    )

    tfidf_matrix = vec.fit_transform(topic_docs)
    feature_names = vec.get_feature_names_out()

    topic_words = {}
    for i, tid in enumerate(tids):
        row = tfidf_matrix.getrow(i).toarray().flatten()
        top_ids = row.argsort()[-top_n:][::-1]
        topic_words[tid] = [feature_names[idx] for idx in top_ids if row[idx] > 0]
    return topic_words


def compute_coherence_irbo(assignment_df: pd.DataFrame, texts: list,
                           max_df: float = 0.8, top_n: int = 10) -> dict:
    """Compute C_v coherence and IRBO diversity with parametric max_df."""
    topic_words = compute_topic_words_ctfidf(assignment_df, texts, max_df, top_n)
    word_lists  = [v for v in topic_words.values() if v]

    tokenized = [t.lower().split() for t in texts]

    # Flatten bigrams to unigrams for gensim
    gensim_word_lists = []
    for words in word_lists:
        topic_unigrams = []
        for w in words:
            topic_unigrams.extend(w.split())
        unique_unigrams = list(dict.fromkeys(topic_unigrams))[:top_n]
        gensim_word_lists.append(unique_unigrams)

    try:
        dct = Dictionary(tokenized)
        cm  = CoherenceModel(
            topics=gensim_word_lists,
            texts=tokenized,
            dictionary=dct,
            coherence="c_v",
            processes=5
        )
        cv = cm.get_coherence()
    except Exception as e:
        print(f"    Coherence error: {e}")
        cv = 0.0
    def irbo(lists1, lists2, p = 0.9):
        return 1 - rbo.RankingSimilarity(lists1, lists2).rbo_ext(p=p)



    pairs = list(combinations(word_lists, 2))
    irbo  = float(np.mean([irbo(a, b) for a, b in pairs])) if pairs else 0.0
    tq    = 2 * cv * irbo / (cv + irbo + 1e-8)

    return {"coherence": cv, "irbo": irbo, "topic_quality": tq}

## Tuning Loop

Grid search over `max_df × min_docs` for each subject.
Reuses the base assignment DataFrames from modeling checkpoints.

In [7]:
all_tuning_results = []

for subject in LIST_SUBJECT:
    if subject not in all_base_assignments:
        print(f"\nSkipping {subject} (no base assignment)")
        continue

    print(f"\n{'='*60}")
    print(f"TUNING: {subject.upper()}")
    print(f"{'='*60}")

    base_df = all_base_assignments[subject]
    texts = all_data[subject]["text"].fillna("").tolist()
    best_model = BEST_MODEL_MAP[subject]

    combos = list(product(MAX_DF_VALUES, MIN_DOCS_VALUES))
    best_tq, best_config = -1, None
    previous_n_topics = 0
    previous_metrics = {}
    prev_max_df = 0.5

    for max_df, min_docs in tqdm(combos, desc=f"Tuning {subject}"):
        ckpt_name = f"tuning_{best_model}_maxdf{max_df}_mindocs{min_docs}"

        # Check if already computed
        ckpt = load_checkpoint(ckpt_name, subject)
        if ckpt:
            metrics = ckpt["metrics"]
            n_topics = ckpt["n_topics"]
        else:
            # Apply filtering with current min_docs
            df_filtered = filter_and_reindex_topics(base_df, min_docs=min_docs)
            n_topics = df_filtered["topic_id"].nunique()

            if n_topics == 0:
                print(f"  max_df={max_df}, min_docs={min_docs}: 0 topics, skipping")
                metrics = {"coherence": 0.0, "irbo": 0.0, "topic_quality": 0.0}
            else:
                # Compute coherence with current max_df
                if previous_n_topics == n_topics and prev_max_df == max_df:
                    metrics = previous_metrics
                else:
                    metrics = compute_coherence_irbo(df_filtered, texts, max_df=max_df)
                previous_n_topics = n_topics
                previous_metrics = metrics
                prev_max_df = max_df

            save_checkpoint(
                {"metrics": metrics, "n_topics": n_topics,
                 "max_df": max_df, "min_docs": min_docs},
                ckpt_name, subject
            )

        row = {
            "subject": subject,
            "best_model": best_model,
            "max_df": max_df,
            "min_docs": min_docs,
            "n_topics": n_topics,
            "coherence": metrics["coherence"],
            "irbo": metrics["irbo"],
            "topic_quality": metrics["topic_quality"],
        }
        all_tuning_results.append(row)

        if metrics["topic_quality"] > best_tq:
            best_tq = metrics["topic_quality"]
            best_config = row

        tqdm.write(
            f"  max_df={max_df:.1f}  min_docs={min_docs:3d}  "
            f"topics={n_topics:3d}  C_v={metrics['coherence']:.4f}  "
            f"IRBO={metrics['irbo']:.4f}  TQ={metrics['topic_quality']:.4f}"
        )

    print(f"\n  BEST for {subject}: max_df={best_config['max_df']}, "
          f"min_docs={best_config['min_docs']}, TQ={best_tq:.4f}")

tuning_df = pd.DataFrame(all_tuning_results)
print(f"\nTotal results: {len(tuning_df)}")


TUNING: CS


Tuning cs:   0%|          | 0/54 [00:00<?, ?it/s]

Tuning cs:   2%|▏         | 1/54 [01:09<1:01:19, 69.43s/it]

  Checkpoint saved: ../../../../models/topicGpt/cs/tuning_sentence_transformers_all_MiniLM_L6_v2_maxdf0.5_mindocs1.pkl
  max_df=0.5  min_docs=  1  topics=138  C_v=0.6799  IRBO=0.9946  TQ=0.8077
  Checkpoint saved: ../../../../models/topicGpt/cs/tuning_sentence_transformers_all_MiniLM_L6_v2_maxdf0.5_mindocs10.pkl
  max_df=0.5  min_docs= 10  topics=138  C_v=0.6799  IRBO=0.9946  TQ=0.8077
  Checkpoint saved: ../../../../models/topicGpt/cs/tuning_sentence_transformers_all_MiniLM_L6_v2_maxdf0.5_mindocs25.pkl
  max_df=0.5  min_docs= 25  topics=138  C_v=0.6799  IRBO=0.9946  TQ=0.8077
  Checkpoint saved: ../../../../models/topicGpt/cs/tuning_sentence_transformers_all_MiniLM_L6_v2_maxdf0.5_mindocs50.pkl
  max_df=0.5  min_docs= 50  topics=138  C_v=0.6799  IRBO=0.9946  TQ=0.8077
  Checkpoint saved: ../../../../models/topicGpt/cs/tuning_sentence_transformers_all_MiniLM_L6_v2_maxdf0.5_mindocs100.pkl
  max_df=0.5  min_docs=100  topics=138  C_v=0.6799  IRBO=0.9946  TQ=0.8077
  Checkpoint saved: ../..

Tuning cs:  15%|█▍        | 8/54 [02:14<11:17, 14.73s/it]  

...

  Checkpoint saved: ../../../../models/topicGpt/physics/tuning_all_distilroberta_v1_maxdf1.0_mindocs1.pkl
  max_df=1.0  min_docs=  1  topics= 74  C_v=0.6903  IRBO=0.9658  TQ=0.8052
  Checkpoint saved: ../../../../models/topicGpt/physics/tuning_all_distilroberta_v1_maxdf1.0_mindocs10.pkl
  max_df=1.0  min_docs= 10  topics= 74  C_v=0.6903  IRBO=0.9658  TQ=0.8052
  Checkpoint saved: ../../../../models/topicGpt/physics/tuning_all_distilroberta_v1_maxdf1.0_mindocs25.pkl
  max_df=1.0  min_docs= 25  topics= 74  C_v=0.6903  IRBO=0.9658  TQ=0.8052
  Checkpoint saved: ../../../../models/topicGpt/physics/tuning_all_distilroberta_v1_maxdf1.0_mindocs50.pkl
  max_df=1.0  min_docs= 50  topics= 74  C_v=0.6903  IRBO=0.9658  TQ=0.8052
  Checkpoint saved: ../../../../models/topicGpt/physics/tuning_all_distilroberta_v1_maxdf1.0_mindocs100.pkl
  max_df=1.0  min_docs=100  topics= 74  C_v=0.6903  IRBO=0.9658  TQ=0.8052
  Checkpoint saved: ../../../../models/topicGpt/physics/tuning_all_distilroberta_v1_maxdf1

Tuning physics:  98%|█████████▊| 53/54 [09:07<00:08,  8.94s/it]

  Checkpoint saved: ../../../../models/topicGpt/physics/tuning_all_distilroberta_v1_maxdf1.0_mindocs250.pkl
  max_df=1.0  min_docs=250  topics= 50  C_v=0.6831  IRBO=0.9586  TQ=0.7977


Tuning physics: 100%|██████████| 54/54 [09:38<00:00, 10.71s/it]

  Checkpoint saved: ../../../../models/topicGpt/physics/tuning_all_distilroberta_v1_maxdf1.0_mindocs300.pkl
  max_df=1.0  min_docs=300  topics= 41  C_v=0.6990  IRBO=0.9591  TQ=0.8086

  BEST for physics: max_df=0.6, min_docs=300, TQ=0.8523

Total results: 162


## Results Summary

In [8]:
# Save full results CSV per subject
for subject in LIST_SUBJECT:
    subj_df = tuning_df[tuning_df["subject"] == subject]
    if len(subj_df) == 0:
        continue
    out_path = RESULT_DIR / subject / "tuning_results.csv"
    subj_df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")

# Also save combined results
combined_path = RESULT_DIR / "tuning_results_all.csv"
tuning_df.to_csv(combined_path, index=False)
print(f"\nCombined results saved: {combined_path}")

# Show best per subject
print(f"\n{'='*80}")
print(f"BEST CONFIGURATION PER SUBJECT")
print(f"{'='*80}")
for subject in LIST_SUBJECT:
    subj_df = tuning_df[tuning_df["subject"] == subject]
    if len(subj_df) == 0:
        continue
    best = subj_df.loc[subj_df["topic_quality"].idxmax()]
    print(f"\n{subject.upper()}:")
    print(f"  Model:    {best['best_model']}")
    print(f"  max_df:   {best['max_df']}")
    print(f"  min_docs: {int(best['min_docs'])}")
    print(f"  Topics:   {int(best['n_topics'])}")
    print(f"  C_v:      {best['coherence']:.4f}")
    print(f"  IRBO:     {best['irbo']:.4f}")
    print(f"  TQ:       {best['topic_quality']:.4f}")

# Pivot table: TQ by max_df x min_docs (averaged across subjects)
print(f"\n{'='*80}")
print(f"TOPIC QUALITY HEATMAP (averaged across subjects)")
print(f"{'='*80}")
pivot = tuning_df.pivot_table(
    index="min_docs", columns="max_df",
    values="topic_quality", aggfunc="mean"
)
print(pivot.round(4))

Saved: ../../../../results/topicGpt/tunning/cs/tuning_results.csv
Saved: ../../../../results/topicGpt/tunning/math/tuning_results.csv
Saved: ../../../../results/topicGpt/tunning/physics/tuning_results.csv

Combined results saved: ../../../../results/topicGpt/tunning/tuning_results_all.csv

BEST CONFIGURATION PER SUBJECT

CS:
  Model:    sentence_transformers_all_MiniLM_L6_v2
  max_df:   0.8
  min_docs: 1
  Topics:   138
  C_v:      0.7095
  IRBO:     0.9893
  TQ:       0.8264

MATH:
  Model:    sentence_transformers_all_MiniLM_L6_v2
  max_df:   0.9
  min_docs: 300
  Topics:   63
  C_v:      0.7468
  IRBO:     0.9799
  TQ:       0.8476

PHYSICS:
  Model:    all_distilroberta_v1
  max_df:   0.6
  min_docs: 300
  Topics:   41
  C_v:      0.7539
  IRBO:     0.9803
  TQ:       0.8523

TOPIC QUALITY HEATMAP (averaged across subjects)
max_df       0.5     0.6     0.7     0.8     0.9     1.0
min_docs                                                
1         0.8195  0.8266  0.8307  0.8334  0.83

## Save Best Tuning Model

For each subject, save the best `(max_df, min_docs)` assignment + config
as a checkpoint and export the final assignment CSV.

In [9]:
for subject in LIST_SUBJECT:
    if subject not in all_base_assignments:
        continue

    subj_df = tuning_df[tuning_df["subject"] == subject]
    if len(subj_df) == 0:
        continue

    best = subj_df.loc[subj_df["topic_quality"].idxmax()]
    best_max_df = best["max_df"]
    best_min_docs = int(best["min_docs"])
    best_model = best["best_model"]

    print(f"\n{'='*60}")
    print(f"SAVING BEST FOR: {subject.upper()}")
    print(f"  model={best_model}, max_df={best_max_df}, min_docs={best_min_docs}")
    print(f"{'='*60}")

    # Re-apply filtering with best params
    base_df = all_base_assignments[subject]
    df_best = filter_and_reindex_topics(base_df, min_docs=best_min_docs)

    # Recompute metrics for verification
    texts = all_data[subject]["text"].fillna("").tolist()
    metrics = compute_coherence_irbo(df_best, texts, max_df=best_max_df)

    print(f"  Verified: C_v={metrics['coherence']:.4f}  "
          f"IRBO={metrics['irbo']:.4f}  TQ={metrics['topic_quality']:.4f}")
    print(f"  Topics: {df_best['topic_id'].nunique()}  Docs: {len(df_best)}")

    # Save best model checkpoint
    save_checkpoint(
        {
            "assignment_df": df_best,
            "metrics": metrics,
            "config": {
                "best_model": best_model,
                "max_df": best_max_df,
                "min_docs": best_min_docs,
            }
        },
        "tuning_best", subject
    )

    # Export assignment CSV
    df_export = df_best.copy()
    df_export["subject"]    = subject
    df_export["best_model"] = best_model
    df_export["max_df"]     = best_max_df
    df_export["min_docs"]   = best_min_docs
    df_export["coherence"]  = metrics["coherence"]
    df_export["irbo"]       = metrics["irbo"]

    out_csv = RESULT_DIR / subject / "topicgpt_assignments.csv"
    df_export.to_csv(out_csv, index=False)
    print(f"  Saved CSV: {out_csv}")
    print(f"    Columns: {list(df_export.columns)}")
    print(f"    Rows: {len(df_export)}")


SAVING BEST FOR: CS
  model=sentence_transformers_all_MiniLM_L6_v2, max_df=0.8, min_docs=1
  Verified: C_v=0.7095  IRBO=0.9893  TQ=0.8264
  Topics: 138  Docs: 65516
  Checkpoint saved: ../../../../models/topicGpt/cs/tuning_best.pkl
  Saved CSV: ../../../../results/topicGpt/tunning/cs/topicgpt_assignments.csv
    Columns: ['doc_idx', 'topic_id', 'topic_label', 'confidence', 'original_topic_id', 'subject', 'best_model', 'max_df', 'min_docs', 'coherence', 'irbo']
    Rows: 65516

SAVING BEST FOR: MATH
  model=sentence_transformers_all_MiniLM_L6_v2, max_df=0.9, min_docs=300
  Verified: C_v=0.7468  IRBO=0.9799  TQ=0.8476
  Topics: 63  Docs: 31811
  Checkpoint saved: ../../../../models/topicGpt/math/tuning_best.pkl
  Saved CSV: ../../../../results/topicGpt/tunning/math/topicgpt_assignments.csv
    Columns: ['doc_idx', 'topic_id', 'topic_label', 'confidence', 'original_topic_id', 'subject', 'best_model', 'max_df', 'min_docs', 'coherence', 'irbo']
    Rows: 31811

SAVING BEST FOR: PHYSICS
  m

## Final Summary

In [10]:
print(f"\n{'='*80}")
print(f"{'SUBJECT':<12} | {'MODEL':<45} | {'max_df':>6} | {'min_docs':>8} | {'TOPICS':>6} | {'TQ':>6}")
print(f"{'-'*80}")

for subject in LIST_SUBJECT:
    subj_df = tuning_df[tuning_df["subject"] == subject]
    if len(subj_df) == 0:
        continue
    best = subj_df.loc[subj_df["topic_quality"].idxmax()]
    print(f"{subject:<12} | {best['best_model']:<45} | {best['max_df']:>6.1f} | {int(best['min_docs']):>8} | {int(best['n_topics']):>6} | {best['topic_quality']:>6.4f}")

print(f"\nDone! All results saved to {RESULT_DIR.resolve()}")
print(f"Best model checkpoints saved to {CHECKPOINT_DIR.resolve()}/{{subject}}/tuning_best.pkl")


SUBJECT      | MODEL                                         | max_df | min_docs | TOPICS |     TQ
--------------------------------------------------------------------------------
cs           | sentence_transformers_all_MiniLM_L6_v2        |    0.8 |        1 |    138 | 0.8264
math         | sentence_transformers_all_MiniLM_L6_v2        |    0.9 |      300 |     63 | 0.8476
physics      | all_distilroberta_v1                          |    0.6 |      300 |     41 | 0.8523

Done! All results saved to /home/nedo/Kuliah/TA/Program/results/topicGpt/tunning
Best model checkpoints saved to /home/nedo/Kuliah/TA/Program/models/topicGpt/{subject}/tuning_best.pkl
